In [ ]:
# Cell 1: imports, config, constants
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from statsmodels.tsa.stattools import acf
import statsmodels.api as sm
from scipy import stats

FILE_PATH = 'fichinH125.xlsx'

SESSION_START  = pd.Timedelta('9:00:00')
SESSION_END    = pd.Timedelta('13:30:00')
BIN_MINUTES    = 15
N_BINS         = 18

BIN_LABEL_STRS = [f'{9 + 15*i//60:02d}:{15*i%60:02d}' for i in range(N_BINS)]
WEEKDAY_ORDER  = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes']
BUCKET_ORDER   = ['neither', 'tax_season', 'month_end']
BUCKET_COLORS  = {'neither': 'steelblue', 'tax_season': 'darkorange', 'month_end': 'seagreen'}

In [ ]:
# Cell 2: pure functions

def load_data(path):
    df = pd.read_excel(path)
    if df['Fecha'].dtype == 'object':
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)
    else:
        df['Fecha'] = pd.to_datetime(df['Fecha']).dt.normalize()
    def to_td(t):
        if isinstance(t, str):
            return pd.to_timedelta(t)
        if hasattr(t, 'hour'):
            return pd.Timedelta(hours=t.hour, minutes=t.minute, seconds=t.second)
        return pd.to_timedelta(t)
    df['Tiempo_td'] = df['Tiempo'].apply(to_td)
    df['Timestamp'] = df['Fecha'] + df['Tiempo_td']
    return df
def calendar_bucket(date):
    # neither: DOM<12  tax_season: DOM 12-24  month_end: DOM>=25
    dom = date.day
    if dom < 12:
        return 'neither'
    if dom <= 24:
        return 'tax_season'
    return 'month_end'
def assign_bin(tiempo_td):
    # -1 if outside [9:00, 13:30)
    if tiempo_td < SESSION_START or tiempo_td >= SESSION_END:
        return -1
    return int((tiempo_td - SESSION_START).total_seconds() // 60) // BIN_MINUTES
def build_bin_volume(df):
    df = df.copy()
    df['bin_idx'] = df['Tiempo_td'].apply(assign_bin)
    in_sess = df[df['bin_idx'] >= 0]
    bv = in_sess.groupby(['Fecha', 'bin_idx'])['Monto'].sum().reset_index()
    idx = pd.MultiIndex.from_product(
        [df['Fecha'].unique(), range(N_BINS)], names=['Fecha', 'bin_idx']
    )
    bv = bv.set_index(['Fecha', 'bin_idx']).reindex(idx, fill_value=0).reset_index()
    bv['bin_label'] = bv['bin_idx'].apply(lambda i: BIN_LABEL_STRS[i])
    return bv
def compute_session_totals(bv):
    return bv.groupby('Fecha')['Monto'].sum().rename('V')
def compute_shares(bv, session_totals):
    bv = bv.copy()
    bv['V'] = bv['Fecha'].map(session_totals)
    bv['share'] = np.where(bv['V'] > 0, bv['Monto'] / bv['V'], 0.0)
    return bv
def profile_stats(bv_shares):
    g = bv_shares.groupby('bin_idx')['share']
    return pd.DataFrame({
        'bin_label': BIN_LABEL_STRS,
        'mean':   g.mean().values,
        'median': g.median().values,
        'q25':    g.quantile(0.25).values,
        'q75':    g.quantile(0.75).values,
    })
def weekday_profiles(bv_shares):
    bv = bv_shares.copy()
    bv['dow'] = bv['Fecha'].map(
        lambda d: ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', None, None][d.weekday()]
    )
    return bv.groupby(['dow', 'bin_idx'])['share'].mean().unstack(level=0)
def bucket_profiles(bv_shares):
    bv = bv_shares.copy()
    bv['bucket'] = bv['Fecha'].apply(calendar_bucket)
    return bv.groupby(['bucket', 'bin_idx'])['share'].mean().unstack(level=0)
def compute_acf_series(series, nlags):
    vals, ci = acf(series, nlags=nlags, alpha=0.05, fft=True)
    return vals[1:], ci[1:]
def pool_within_session_acf(bv_shares, nlags=5):
    mu = bv_shares.groupby('bin_idx')['share'].mean()
    bv = bv_shares.copy()
    bv['mu'] = bv['bin_idx'].map(mu)
    bv['u'] = np.where(bv['mu'] > 0, bv['share'] / bv['mu'], np.nan)
    dates = sorted(bv['Fecha'].unique())
    lag_prods = {lag: [] for lag in range(1, nlags + 1)}
    for d in dates:
        u_s = bv[bv['Fecha'] == d].sort_values('bin_idx')['u'].values
        for lag in range(1, nlags + 1):
            lag_prods[lag].extend(zip(u_s[lag:], u_s[:-lag]))
    u_all = bv['u'].dropna().values
    u_mean, u_var = np.nanmean(u_all), np.nanvar(u_all)
    acf_vals = {}
    for lag, pairs in lag_prods.items():
        p = [(a, b) for a, b in pairs if not (np.isnan(a) or np.isnan(b))]
        if p:
            aa = np.array([x[0] for x in p])
            ba = np.array([x[1] for x in p])
            acf_vals[lag] = np.mean((aa - u_mean) * (ba - u_mean)) / u_var
        else:
            acf_vals[lag] = np.nan
    return acf_vals, 1.96 / np.sqrt(len(u_all))
def fit_ar_regression(log_v_series):
    # AR(2) + weekday dummies (Mon ref) + tax_season + month_end dummies
    s = log_v_series.copy()
    df_r = pd.DataFrame({
        'logV':  s.values,
        'Fecha': s.index,
        'lag1':  np.concatenate([[np.nan], s.values[:-1]]),
        'lag2':  np.concatenate([[np.nan, np.nan], s.values[:-2]]),
    })
    df_r['bucket'] = df_r['Fecha'].apply(calendar_bucket)
    df_r['dow']    = df_r['Fecha'].apply(lambda d: d.weekday())
    df_r = df_r.dropna(subset=['lag1', 'lag2'])
    for i, name in enumerate(['tue', 'wed', 'thu', 'fri'], start=1):
        df_r[f'wd_{name}'] = (df_r['dow'] == i).astype(float)
    df_r['tax_season'] = (df_r['bucket'] == 'tax_season').astype(float)
    df_r['month_end']  = (df_r['bucket'] == 'month_end').astype(float)
    feat = ['lag1', 'lag2', 'wd_tue', 'wd_wed', 'wd_thu', 'wd_fri', 'tax_season', 'month_end']
    return sm.OLS(df_r['logV'], sm.add_constant(df_r[feat])).fit()
def flag_outlier_sessions(sess_tot, bv_shares, sigma_thresh=2.5, share_thresh=0.30):
    # within-bucket z-score for V(d) outliers
    st = sess_tot.reset_index()
    st.columns = ['Fecha', 'V']
    st['bucket'] = st['Fecha'].apply(calendar_bucket)
    st['logV'] = np.log(st['V'])
    flags = []
    for bkt, grp in st.groupby('bucket'):
        if len(grp) < 3:
            continue
        mu, sd = grp['logV'].mean(), grp['logV'].std()
        if sd == 0 or np.isnan(sd):
            continue
        for _, row in grp.iterrows():
            z = (row['logV'] - mu) / sd
            if abs(z) > sigma_thresh:
                flags.append({'date': row['Fecha'], 'bucket': bkt,
                              'reason': f'V outlier within {bkt} (z={z:.2f})', 'stat': row['V']})
    for d, ms in bv_shares.groupby('Fecha')['share'].max().items():
        if ms > share_thresh:
            flags.append({'date': d, 'bucket': calendar_bucket(d),
                          'reason': f'bin share > {share_thresh} (max={ms:.3f})', 'stat': ms})
    return (pd.DataFrame(flags).sort_values('date') if flags
            else pd.DataFrame(columns=['date', 'bucket', 'reason', 'stat']))
def print_summary(label, text):
    print(f'\n--- {label} ---')
    print(text)

In [ ]:
# Cell 3: execution

# ── load ──────────────────────────────────────────────────────────────────────
raw = load_data(FILE_PATH)
print('Loaded:', len(raw), 'rows')
print('Columns:', raw.columns.tolist())
print('Date range:', raw['Fecha'].min().date(), '→', raw['Fecha'].max().date())

by_date = raw.groupby('Fecha')['Tiempo_td'].agg(['min', 'max'])
outside = by_date[(by_date['min'] < SESSION_START) | (by_date['max'] > SESSION_END)]
print('\nTimestamp sanity — dates with trades outside 9:00–13:30:')
print(outside if len(outside) else '  none')

reg_counts = raw['Reg'].value_counts(dropna=False)
print('\nReg value counts and share:')
print(pd.DataFrame({'count': reg_counts, 'share': (reg_counts / len(raw)).round(4)}))
print('\nMonto stats by Reg:')
print(raw.groupby('Reg')['Monto'].describe().round(0))

# set DROP_R = True after inspecting output above
DROP_R = False
working = raw[raw['Reg'] != 'R'].copy() if DROP_R else raw.copy()
print(f'\nDROP_R={DROP_R}  →  working set: {len(working)} rows')

# ── bin construction ──────────────────────────────────────────────────────────
dropped_1330 = (working['Tiempo_td'] == SESSION_END).sum()
print(f'Rows at exactly 13:30:00 (dropped): {dropped_1330}')

bv        = build_bin_volume(working)
sess_tot  = compute_session_totals(bv)
bv_s      = compute_shares(bv, sess_tot)
x         = np.arange(N_BINS)
print(f'Sessions: {sess_tot.shape[0]}  |  bin rows: {bv.shape[0]}')

# ── calendar flags ────────────────────────────────────────────────────────────
bucket_map = {d: calendar_bucket(d) for d in sess_tot.index}
sess_bkt   = pd.Series(bucket_map, name='bucket')
bkt_counts = sess_bkt.value_counts().reindex(BUCKET_ORDER, fill_value=0)
print('\nSessions per calendar bucket:')
print(bkt_counts.to_string())

bv['bucket']  = bv['Fecha'].map(bucket_map)
bv_s['bucket'] = bv_s['Fecha'].map(bucket_map)

# ══════════════════════════════════════════════════════════════════════════════
# D1. Average intraday volume profile
# ══════════════════════════════════════════════════════════════════════════════
prof = profile_stats(bv_s)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x, prof['mean'], color='steelblue', alpha=0.7, label='Mean share')
ax.fill_between(x, prof['q25'], prof['q75'], alpha=0.25, color='steelblue', label='IQR')
ax.plot(x, prof['median'], 'o-', color='black', lw=1.2, ms=4, label='Median')
ax.set_xticks(x)
ax.set_xticklabels(BIN_LABEL_STRS, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Share of daily volume')
ax.set_title('D1 — Average intraday volume profile')
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
plt.tight_layout()
plt.savefig('d1_profile.png', dpi=120)
plt.show()

flat_share   = 1 / N_BINS
max_dev_flat = (prof['mean'] - flat_share).abs().max()
print('\nD1 μ(t):')
print(prof[['bin_label', 'mean', 'median']].to_string(index=False))
print(f'Uniform share = {flat_share:.4f}  |  max |μ(t) - 1/18| = {max_dev_flat:.4f}')
shape_flag = 'visibly non-flat' if max_dev_flat > 0.01 else 'approximately flat'
print(f'Shape: {shape_flag}')

# ══════════════════════════════════════════════════════════════════════════════
# D2. Stability of diurnal shape
# ══════════════════════════════════════════════════════════════════════════════
wd_prof  = weekday_profiles(bv_s)
bkt_prof = bucket_profiles(bv_s)

sess_df       = pd.DataFrame({'V': sess_tot, 'bucket': sess_bkt})
bkt_vol_stats = sess_df.groupby('bucket')['V'].agg(['mean', 'median', 'std', 'count'])
bkt_vol_stats['se'] = bkt_vol_stats['std'] / np.sqrt(bkt_vol_stats['count'])
bkt_vol_stats = bkt_vol_stats.reindex(BUCKET_ORDER)

y_max_ab = max(
    max(wd_prof[c].max() for c in wd_prof.columns),
    max(bkt_prof[c].max() for c in bkt_prof.columns)
) * 1.1

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

ax = axes[0]
for wd in WEEKDAY_ORDER:
    if wd in wd_prof.columns:
        ax.plot(x, wd_prof[wd].values, marker='.', lw=1.2, ms=4, label=wd)
ax.set_xticks(x); ax.set_xticklabels(BIN_LABEL_STRS, rotation=45, ha='right', fontsize=7)
ax.set_ylim(0, y_max_ab); ax.set_ylabel('Mean share'); ax.set_title('D2A — by weekday')
ax.legend(fontsize=7); ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

ax = axes[1]
for bkt in BUCKET_ORDER:
    if bkt in bkt_prof.columns:
        ax.plot(x, bkt_prof[bkt].values, marker='.', lw=1.5, ms=4,
                color=BUCKET_COLORS[bkt], label=bkt)
ax.set_xticks(x); ax.set_xticklabels(BIN_LABEL_STRS, rotation=45, ha='right', fontsize=7)
ax.set_ylim(0, y_max_ab); ax.set_ylabel('Mean share'); ax.set_title('D2B — by calendar bucket')
ax.legend(fontsize=7); ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

ax = axes[2]
bkt_present = [b for b in BUCKET_ORDER if b in bkt_vol_stats.index and not np.isnan(bkt_vol_stats.loc[b, 'mean'])]
ax.bar(bkt_present,
       [bkt_vol_stats.loc[b, 'mean'] for b in bkt_present],
       yerr=[bkt_vol_stats.loc[b, 'se'] for b in bkt_present],
       capsize=4, color=[BUCKET_COLORS[b] for b in bkt_present], alpha=0.75, ecolor='black')
ax.set_ylabel('Mean V(d) USD'); ax.set_title('D2C — Mean session volume by bucket')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('d2_stability.png', dpi=120)
plt.show()

wd_arr  = np.column_stack([wd_prof[w].values for w in WEEKDAY_ORDER if w in wd_prof.columns])
wd_max_dev  = float(np.nanmax(np.abs(wd_arr[:, :, None] - wd_arr[:, None, :])))
bkt_arr = np.column_stack([bkt_prof[b].values for b in BUCKET_ORDER if b in bkt_prof.columns])
bkt_max_dev = float(np.nanmax(np.abs(bkt_arr[:, :, None] - bkt_arr[:, None, :])))
mean_V      = bkt_vol_stats['mean']
v_ratio_tax  = mean_V.get('tax_season', np.nan) / mean_V.get('neither', np.nan)
v_ratio_mend = mean_V.get('month_end',  np.nan) / mean_V.get('neither', np.nan)

print(f'\nD2 weekday max abs deviation (any bin):          {wd_max_dev:.4f}')
print(f'D2 calendar-bucket max abs deviation (any bin): {bkt_max_dev:.4f}')
print('\nD2 V(d) by calendar bucket:')
print(bkt_vol_stats[['mean', 'median', 'std', 'count', 'se']].round(0).to_string())
print(f'\nRatio mean V(tax_season) / mean V(neither): {v_ratio_tax:.3f}')
print(f'Ratio mean V(month_end)  / mean V(neither): {v_ratio_mend:.3f}')

# ══════════════════════════════════════════════════════════════════════════════
# D3. Persistence of V(d)
# ══════════════════════════════════════════════════════════════════════════════
log_v = np.log(sess_tot)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(log_v.values, lw=0.7, color='grey', zorder=1)
for bkt in BUCKET_ORDER:
    mask = sess_bkt == bkt
    pos  = np.where(mask.values)[0]
    ax.scatter(pos, log_v.values[pos], s=15, color=BUCKET_COLORS[bkt], label=bkt, zorder=2)
ax.set_xlabel('Session index'); ax.set_ylabel('log V(d)')
ax.set_title('D3 — log V(d) time series'); ax.legend(fontsize=7)

acf_vals_v, _ = compute_acf_series(log_v.values, nlags=10)
lags_v = np.arange(1, 11)
ci_v   = 1.96 / np.sqrt(len(log_v))

ax = axes[1]
ax.bar(lags_v, acf_vals_v, color='steelblue', alpha=0.7)
ax.axhline(ci_v,  ls='--', color='red', lw=1, label='95% CI')
ax.axhline(-ci_v, ls='--', color='red', lw=1)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('Lag'); ax.set_ylabel('ACF')
ax.set_title('D3 — ACF of log V(d)'); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('d3_persistence.png', dpi=120)
plt.show()

res_ar  = fit_ar_regression(log_v)
ts_coef = res_ar.params.get('tax_season', np.nan)
ts_t    = res_ar.tvalues.get('tax_season', np.nan)
me_coef = res_ar.params.get('month_end', np.nan)
me_t    = res_ar.tvalues.get('month_end', np.nan)

print('\nD3 AR regression (log V ~ const + lag1 + lag2 + wd_dummies + tax_season + month_end):')
print(res_ar.summary2().tables[1][['Coef.', 'Std.Err.', 't', 'P>|t|']].to_string())
print(f'R² = {res_ar.rsquared:.4f}')
print(f'\ntax_season: coef={ts_coef:+.4f}, t={ts_t:.2f}')
print(f'month_end:  coef={me_coef:+.4f}, t={me_t:.2f}')
print('\nD3 ACF of log V(d):')
for lag, val in zip(lags_v, acf_vals_v):
    print(f'  lag {lag:2d}: {val:+.4f}  {"*" if abs(val) > ci_v else ""}')

# ══════════════════════════════════════════════════════════════════════════════
# D4. Within-session ACF of detrended residual
# ══════════════════════════════════════════════════════════════════════════════
u_acf, u_ci = pool_within_session_acf(bv_s, nlags=5)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(list(u_acf.keys()), list(u_acf.values()), color='steelblue', alpha=0.7)
ax.axhline(u_ci,  ls='--', color='red', lw=1, label='95% CI')
ax.axhline(-u_ci, ls='--', color='red', lw=1)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('Lag (bins within session)'); ax.set_ylabel('ACF')
ax.set_title('D4 — Within-session ACF of u(t,d)'); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('d4_acf_u.png', dpi=120)
plt.show()

print('\nD4 ACF of u(t,d) within session:')
for lag, val in u_acf.items():
    print(f'  lag {lag}: {val:+.4f}  {"*" if abs(val) > u_ci else ""}')
print(f'95% CI bound: ±{u_ci:.4f}')

# ══════════════════════════════════════════════════════════════════════════════
# D5. Distribution of V(d)
# ══════════════════════════════════════════════════════════════════════════════
v_ser  = sess_tot
lv_ser = np.log(v_ser)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for bkt in BUCKET_ORDER:
    mask = sess_bkt == bkt
    axes[0].hist(v_ser[mask].values,  bins=20, alpha=0.5, color=BUCKET_COLORS[bkt],
                 label=bkt, edgecolor='white')
    axes[1].hist(lv_ser[mask].values, bins=20, alpha=0.5, color=BUCKET_COLORS[bkt],
                 label=bkt, edgecolor='white')
axes[0].set_xlabel('V(d)');      axes[0].set_title('D5 — Histogram V(d) by bucket');      axes[0].legend(fontsize=7)
axes[1].set_xlabel('log V(d)');  axes[1].set_title('D5 — Histogram log V(d) by bucket'); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig('d5_hist.png', dpi=120)
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
sm.qqplot(lv_ser.values, line='s', ax=axes[0], alpha=0.5)
axes[0].set_title('D5 QQ — Overall')
for i, bkt in enumerate(BUCKET_ORDER):
    sm.qqplot(lv_ser[sess_bkt == bkt].values, line='s', ax=axes[i + 1], alpha=0.5)
    axes[i + 1].set_title(f'D5 QQ — {bkt}')
plt.tight_layout()
plt.savefig('d5_qq.png', dpi=120)
plt.show()

print('\nD5 stats — overall:')
print(f'  V:    mean={v_ser.mean():.0f}  std={v_ser.std():.0f}  '
      f'skew={stats.skew(v_ser.values):.3f}  kurt={stats.kurtosis(v_ser.values):.3f}')
print(f'  logV: mean={lv_ser.mean():.4f}  std={lv_ser.std():.4f}  '
      f'skew={stats.skew(lv_ser.values):.3f}  kurt={stats.kurtosis(lv_ser.values):.3f}')
print('\nD5 stats — by bucket:')
for bkt in BUCKET_ORDER:
    mask = sess_bkt == bkt
    vb, lvb = v_ser[mask].values, lv_ser[mask].values
    print(f'  {bkt} (n={mask.sum()}):')
    print(f'    V:    mean={vb.mean():.0f}  std={vb.std():.0f}  '
          f'skew={stats.skew(vb):.3f}  kurt={stats.kurtosis(vb):.3f}')
    print(f'    logV: mean={lvb.mean():.4f}  std={lvb.std():.4f}  '
          f'skew={stats.skew(lvb):.3f}  kurt={stats.kurtosis(lvb):.3f}')

# ══════════════════════════════════════════════════════════════════════════════
# D6. Coefficient of variation by bin
# ══════════════════════════════════════════════════════════════════════════════
cv_overall = bv.groupby('bin_idx')['Monto'].apply(lambda s: s.std() / s.mean())

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x, cv_overall.values, color='steelblue', alpha=0.55, label='Overall')
for bkt in BUCKET_ORDER:
    cv_b = (bv[bv['bucket'] == bkt]
            .groupby('bin_idx')['Monto']
            .apply(lambda s: s.std() / s.mean()))
    ax.plot(x, cv_b.values, marker='.', lw=1.2, ms=4, color=BUCKET_COLORS[bkt], label=bkt)
ax.set_xticks(x); ax.set_xticklabels(BIN_LABEL_STRS, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('CV'); ax.set_title('D6 — CV by bin (overall + by bucket)')
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig('d6_cv.png', dpi=120)
plt.show()

print('\nD6 CV by bin — overall:')
for lbl, cv in zip(BIN_LABEL_STRS, cv_overall.values):
    print(f'  {lbl}: {cv:.4f}')
print('\nD6 CV by bin — by bucket:')
for bkt in BUCKET_ORDER:
    cv_b = (bv[bv['bucket'] == bkt]
            .groupby('bin_idx')['Monto']
            .apply(lambda s: s.std() / s.mean()))
    print(f'  {bkt}: ' + '  '.join(f'{BIN_LABEL_STRS[i]}={v:.3f}' for i, v in enumerate(cv_b.values)))

# ══════════════════════════════════════════════════════════════════════════════
# D7. Outlier / regime sessions
# ══════════════════════════════════════════════════════════════════════════════
flags      = flag_outlier_sessions(sess_tot, bv_s)
lv_arr     = np.log(sess_tot.values)
dates_all  = sess_tot.index

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(lv_arr, lw=0.7, color='grey', zorder=1)
for bkt in BUCKET_ORDER:
    mask = np.array([bucket_map.get(d) == bkt for d in dates_all])
    pos  = np.where(mask)[0]
    ax.scatter(pos, lv_arr[mask], s=18, color=BUCKET_COLORS[bkt], label=bkt, zorder=2, alpha=0.85)
    if pos.size:
        mu_b, sd_b = lv_arr[mask].mean(), lv_arr[mask].std()
        ax.hlines([mu_b + 2.5*sd_b, mu_b - 2.5*sd_b],
                  pos.min(), pos.max(), colors=BUCKET_COLORS[bkt], linestyles=':', lw=1)

if len(flags):
    flagged_pos = []
    for d in flags['date'].unique():
        idx = np.where(dates_all == d)[0]
        flagged_pos.extend(idx.tolist())
    ax.scatter(flagged_pos, lv_arr[flagged_pos], s=80, marker='x',
               color='red', zorder=3, linewidths=1.5, label='Flagged')

ax.set_xlabel('Session index'); ax.set_ylabel('log V(d)')
ax.set_title('D7 — log V(d) by bucket, dotted lines = within-bucket ±2.5σ')
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig('d7_outliers.png', dpi=120)
plt.show()

print('\nD7 Flagged sessions:')
print(flags.to_string(index=False) if len(flags) else '  none')

# ══════════════════════════════════════════════════════════════════════════════
# Summaries
# ══════════════════════════════════════════════════════════════════════════════
print_summary('D1 Summary',
    f'The average intraday profile across {len(sess_tot)} sessions is {shape_flag}. '
    f'Peak bin: {prof.loc[prof["mean"].idxmax(), "bin_label"]} (μ={prof["mean"].max():.4f}); '
    f'trough: {prof.loc[prof["mean"].idxmin(), "bin_label"]} (μ={prof["mean"].min():.4f}). '
    f'Max deviation from uniform (1/18={flat_share:.4f}): {max_dev_flat:.4f}.'
)

print_summary('D2 Summary',
    f'Weekday max abs shape deviation: {wd_max_dev:.4f}. '
    f'Calendar-bucket max abs shape deviation: {bkt_max_dev:.4f}. '
    f'Mean V(d) — neither: {mean_V.get("neither", float("nan")):.0f}, '
    f'tax_season: {mean_V.get("tax_season", float("nan")):.0f} (ratio={v_ratio_tax:.3f}x), '
    f'month_end: {mean_V.get("month_end", float("nan")):.0f} (ratio={v_ratio_mend:.3f}x). '
    f'A ratio above 1.0 for tax_season or month_end indicates a level shift in V(d) relative to neither; '
    f'a non-zero bkt_max_dev indicates a shape shift in μ(t).'
)

acf_sig_v = [int(l) for l, v in zip(lags_v, acf_vals_v) if abs(v) > ci_v]
print_summary('D3 Summary',
    f'log V(d) AR(2)+calendar regression: R²={res_ar.rsquared:.4f}. '
    f'lag-1 coef={res_ar.params["lag1"]:+.4f} (t={res_ar.tvalues["lag1"]:.2f}), '
    f'lag-2 coef={res_ar.params["lag2"]:+.4f} (t={res_ar.tvalues["lag2"]:.2f}). '
    f'tax_season coef={ts_coef:+.4f} (t={ts_t:.2f}); '
    f'month_end coef={me_coef:+.4f} (t={me_t:.2f}). '
    f'ACF significant lags: {acf_sig_v if acf_sig_v else "none"}.'
)

u_sig = [l for l, v in u_acf.items() if abs(v) > u_ci]
print_summary('D4 Summary',
    f'Within-session ACF of u(t,d)=s(t,d)/μ(t), pooled over {len(sess_tot)} sessions '
    f'respecting session boundaries. 95% CI: ±{u_ci:.4f}. '
    f'Significant lags: {u_sig if u_sig else "none"}. '
    f'Values: ' + ', '.join(f'lag {l}={v:+.4f}' for l, v in u_acf.items()) + '.'
)

print_summary('D5 Summary',
    f'Overall — V(d): mean={v_ser.mean():.0f}, std={v_ser.std():.0f}, '
    f'skew={stats.skew(v_ser.values):.3f}, kurt={stats.kurtosis(v_ser.values):.3f}; '
    f'log V(d): mean={lv_ser.mean():.4f}, std={lv_ser.std():.4f}, '
    f'skew={stats.skew(lv_ser.values):.3f}, kurt={stats.kurtosis(lv_ser.values):.3f}. '
    f'By bucket (log V mean): neither={lv_ser[sess_bkt=="neither"].mean():.4f}, '
    f'tax_season={lv_ser[sess_bkt=="tax_season"].mean():.4f}, '
    f'month_end={lv_ser[sess_bkt=="month_end"].mean():.4f}. '
    f'By bucket (log V std): neither={lv_ser[sess_bkt=="neither"].std():.4f}, '
    f'tax_season={lv_ser[sess_bkt=="tax_season"].std():.4f}, '
    f'month_end={lv_ser[sess_bkt=="month_end"].std():.4f}.'
)

cv_mean_bkt = {}
for bkt in BUCKET_ORDER:
    cv_b = (bv[bv['bucket'] == bkt]
            .groupby('bin_idx')['Monto']
            .apply(lambda s: s.std() / s.mean()))
    cv_mean_bkt[bkt] = cv_b.mean()
print_summary('D6 Summary',
    f'Overall CV(t) ranges from {cv_overall.min():.4f} ({BIN_LABEL_STRS[int(cv_overall.idxmin())]}) '
    f'to {cv_overall.max():.4f} ({BIN_LABEL_STRS[int(cv_overall.idxmax())]}); '
    f'mean CV={cv_overall.mean():.4f}. '
    f'Mean CV by bucket: neither={cv_mean_bkt.get("neither", float("nan")):.4f}, '
    f'tax_season={cv_mean_bkt.get("tax_season", float("nan")):.4f}, '
    f'month_end={cv_mean_bkt.get("month_end", float("nan")):.4f}. '
    f'Differences indicate whether dispersion within bins is higher or lower in tax_season or month_end sessions.'
)

flag_lines = ('; '.join(
    f'{row["date"].date()} [{row["bucket"]}] — {row["reason"]}'
    for _, row in flags.iterrows()
) if len(flags) else 'none')
print_summary('D7 Summary',
    f'{len(flags)} flag(s) across {flags["date"].nunique() if len(flags) else 0} session(s). '
    f'V(d) outliers detected within-bucket (|z|>2.5); bin-share outliers unconditional (>0.30). '
    f'Flagged: {flag_lines}.'
)